# Lab 07-01 · Capacitors and Inductors

Companion to chapter [07-01](../../chapters/07-01-capacitors-and-inductors.md). Handout: [README.md](README.md).

Cells marked **🔮 Predict** ask for a prediction *before* running.

In [1]:
import shutil
import subprocess
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HAVE_NGSPICE = shutil.which("ngspice") is not None
print("ngspice available:", HAVE_NGSPICE)


def ngspice_values(path):
    # Run ngspice in batch mode and collect every 'name = value' line it prints.
    out = subprocess.run(["ngspice", "-b", path], capture_output=True, text=True).stdout
    vals = {}
    for line in out.splitlines():
        parts = line.split("=")
        if len(parts) >= 2:
            try:
                vals[parts[0].strip()] = float(parts[1].split()[0])
            except (ValueError, IndexError):
                pass
    return vals


NG = ngspice_values("storage.cir") if HAVE_NGSPICE else {}

ngspice available: True


## Part 1 — Waveforms in and out

Example 07-01.1 drives a 10 µF capacitor with a piecewise-linear voltage; Example 07-01.2 drives a 4.7 µH inductor with a triangular current.

**🔮 Predict:** sketch the capacitor's current and the inductor's voltage before running.

In [2]:
t = np.linspace(0, 3e-3, 30001)
v = np.interp(t, [0, 1e-3, 2e-3, 2.5e-3, 3e-3], [0, 5, 5, 0, 0])
i = 10e-6 * np.gradient(v, t)                       # i = C dv/dt, numerically
tl = np.linspace(0, 10e-6, 10001)
il = np.interp(tl, [0, 2e-6, 5e-6, 7e-6, 10e-6], [1, 3, 1, 3, 1])
vl = 4.7e-6 * np.gradient(il, tl)                   # v = L di/dt, numerically
fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.2))
axes[0].plot(t * 1e3, i * 1e3); axes[0].set_xlabel("time (ms)"); axes[0].set_ylabel("i_C (mA)")
axes[1].plot(tl * 1e6, vl); axes[1].set_xlabel("time (µs)"); axes[1].set_ylabel("v_L (V)")
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print(f"capacitor current: {i[t < 0.9e-3].mean() * 1e3:.2f} mA rising, {i[(t > 2.1e-3) & (t < 2.4e-3)].mean() * 1e3:.2f} mA falling")
print(f"inductor voltage: {vl[tl < 1.9e-6].mean():.3f} V rising, {vl[(tl > 2.1e-6) & (tl < 4.9e-6)].mean():.3f} V falling")
if NG:
    print("ngspice:", NG["ex1a"] * 1e3, "mA (source current, so sign reversed) and", NG["ex2a"], "V,", NG["ex2b"], "V")
    assert abs(-NG["ex1a"] - 0.05) < 1e-6

capacitor current: 50.00 mA rising, -100.00 mA falling
inductor voltage: 4.700 V rising, -3.133 V falling
ngspice: -50.0 mA (source current, so sign reversed) and 4.7 V, -3.133333 V


Wherever the voltage is a straight line, the capacitor current is constant, and the current jumps at every corner: only the *voltage* must be continuous. The inductor is the mirror image.

## Part 2 — Charge redistribution

A 10 µF capacitor at 10 V is connected through a resistor $R$ to an uncharged 10 µF capacitor.

**🔮 Predict:** does the energy lost depend on $R$? What is the peak current for 1 Ω and for 0.01 Ω?

In [3]:
def share(R, C1=10e-6, C2=10e-6, V0=10.0):
    tau = R * C1 * C2 / (C1 + C2)
    def f(t, y):
        i = (y[0] - y[1]) / R
        return [-i / C1, i / C2, i * i * R]          # v1, v2, energy dissipated in R
    sol = solve_ivp(f, (0, 20 * tau), [V0, 0.0, 0.0], rtol=1e-10, atol=1e-14, dense_output=True)
    return sol.y[:, -1], V0 / R

for R in (1.0, 0.01):
    (v1, v2, w_lost), ipk = share(R)
    print(f"R = {R:5.2f} ohm: final {v1:.4f} V and {v2:.4f} V, peak current {ipk:7.1f} A, energy lost {w_lost * 1e6:.2f} µJ")
w0 = 0.5 * 10e-6 * 100
print(f"energy before {w0 * 1e6:.1f} µJ, after {2 * 0.5 * 10e-6 * 25 * 1e6:.1f} µJ")
assert abs(share(0.01)[0][2] - 250e-6) < 1e-9

R =  1.00 ohm: final 5.0000 V and 5.0000 V, peak current    10.0 A, energy lost 250.00 µJ
R =  0.01 ohm: final 5.0000 V and 5.0000 V, peak current  1000.0 A, energy lost 250.00 µJ
energy before 500.0 µJ, after 250.0 µJ


Exactly half the energy, 250 µJ, is lost whatever the resistance: a smaller resistor gives a larger, shorter current pulse with the same total dissipation (Problem 07-01.6). With "zero" resistance the energy goes into radiation and sparks rather than disappearing.

## Part 3 — The inductive kick

**🔮 Predict:** Example 07-01.4's 100 mH coil carries 50 mA. What voltage appears if the current is forced to zero in 1 µs, and what does a flyback diode do?

In [4]:
L, I = 0.1, 0.05
print(f"forced to zero in 1 µs: v = L di/dt = {L * I / 1e-6:.0f} V; stored energy {0.5 * L * I * I * 1e6:.0f} µJ")
if NG:
    print(f"ngspice: {NG['ex4a']:.0f} V without a path; with a flyback diode the switch sees {NG['ex4c']:.2f} V")
    assert abs(NG["ex4a"] + 5000) < 5

forced to zero in 1 µs: v = L di/dt = 5000 V; stored energy 125 µJ
ngspice: -5000 V without a path; with a flyback diode the switch sees 12.67 V


Without a path, the voltage is set only by how fast the current is stopped; with a diode, the coil's current circulates and the switch sees the supply plus one diode drop. 07-02 follows the current as it then decays.

## Part 4 — A real capacitor's impedance

The model: 10 µF in series with 5 mΩ ESR and 0.5 nH ESL.

**🔮 Predict:** where is the impedance minimum, and how does putting 20 of them in parallel change the curve?

In [5]:
f = np.logspace(4, 9, 2000)
w = 2 * np.pi * f
def zcap(C=10e-6, Lesl=0.5e-9, R=5e-3, n=1):
    return np.abs(R / n + 1j * w * Lesl / n + 1 / (1j * w * C * n))
z1, z20 = zcap(), zcap(n=20)
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.loglog(f, z1, label="one capacitor")
ax.loglog(f, z20, label="20 in parallel")
ax.set_xlabel("frequency (Hz)"); ax.set_ylabel("|Z| (Ω)"); ax.grid(True, which="both", alpha=0.3); ax.legend()
plt.show()
f_sr = 1 / (2 * np.pi * np.sqrt(0.5e-9 * 10e-6))
print(f"self-resonance {f_sr / 1e6:.3f} MHz for both; minimum |Z| {z1.min() * 1e3:.2f} mΩ (one) and {z20.min() * 1e3:.3f} mΩ (20)")
if NG:
    print(f"ngspice: phase crosses zero at {NG['ex5e'] / 1e6:.3f} MHz")
    assert abs(NG["ex5e"] - f_sr) / f_sr < 2e-3

self-resonance 2.251 MHz for both; minimum |Z| 5.00 mΩ (one) and 0.250 mΩ (20)
ngspice: phase crosses zero at 2.251 MHz


Paralleling identical parts divides the impedance by $N$ at every frequency but does not move the resonance, because the total inductance falls by $N$ while the capacitance rises by $N$. Mixing values does move resonances, and can create anti-resonant peaks between them (Course 07 frequency response, Course 22 power integrity).

## Your turn

1. In Part 2, connect three 10 µF capacitors (one at 10 V, two empty) in a ring through resistors. What fraction of the energy survives?
2. In Part 4, replace the 10 µF parts with 1 µF parts of the same ESL. Where is the self-resonance now?
3. What ESL would put a 10 µF capacitor's self-resonance at 10 MHz?

<details><summary>Answers</summary>

1. The charge 100 µC spreads over 30 µF, so each ends at 3.33 V; the energy is $3\times\tfrac12\times10\ \text{µF}\times3.33^2 = 167$ µJ of the original 500 µJ: one third survives.
2. $f_{SR}$ rises by $\sqrt{10}$, to 7.118 MHz.
3. $L = 1/[(2\pi f)^2C] = 25.33$ pH, far below any surface-mount part's ESL: only on-package or on-die capacitance gets near it.

</details>